In [1]:
import polars as pl
import pandera.polars as pa
from datetime import datetime
from src.ecommerce_etl import config
from src.ecommerce_etl.io import read_csv_raw, write_parquet
from src.ecommerce_etl.transforms import (
    parse_datetime, strip_strings, lowercase, uppercase,
    cast_float, nullify_outside_range, nullify_not_in_set,
    drop_null_keys, deduplicate_by_key,
)

Customers

In [2]:
customers = read_csv_raw("customers")
s_customers = customers.sample(fraction=0.1, seed=42)
s_customers.head(5)

customer_id,customer_name,email,city,state,created_at
str,str,str,str,str,str
"""CUST-00001""","""Beatriz Pereira""","""beatriz.pereira_1@email.com""","""Salvador""","""BA""","""2022-05-14 01:01:00"""
"""CUST-00003""","""Diego Costa""","""diego.costa_3@email.com""","""Campinas""","""SP""","""2022-01-05 11:38:00"""
"""CUST-00004""","""Patricia Cardoso""","""patricia.cardoso_4@email.com""","""Belem""","""PA""","""2022-01-10 18:12:00"""
"""CUST-00008""","""Larissa Araujo""","""larissa.araujo_8@email.com""","""Caruaru""","""PE""","""2022-05-27 18:25:00"""
"""CUST-00006""","""Beatriz Oliveira""","""beatriz.oliveira_6@email.com""","""Goiania""","""GO""","""2023-08-03 22:35:00"""


In [3]:
class Customers(pa.DataFrameModel):
    customer_id: str = pa.Field(nullable=False, unique=True)
    customer_name: str = pa.Field(nullable=True)
    email: str = pa.Field(nullable=True)
    city: str = pa.Field(nullable=True)
    state: str = pa.Field(nullable=True)
    created_at: datetime = pa.Field(nullable=False)
    source_file: str = pa.Field(alias="_source_file", nullable=False)
    ingested_at: datetime = pa.Field(alias="_ingested_at", nullable=False)

    class Config:
        coerce = False
        strict = True

In [4]:
customers_bronze = s_customers.with_columns(
    pl.lit("customers.csv").alias("_source_file"),
    pl.lit(datetime.now()).alias("_ingested_at")
)
write_parquet(customers_bronze, config.BRONZE_DIR / "customers.parquet")

In [5]:
STRING_COLS = ["customer_id", "customer_name", "email", "city", "state"]

customers = parse_datetime(customers_bronze, "created_at")
customers = strip_strings(customers, STRING_COLS)
customers = lowercase(customers, "email")
customers = uppercase(customers, "state")

customers, rej_null = drop_null_keys(customers, "customer_id")
customers, rej_dup = deduplicate_by_key(customers, "customer_id", "created_at")

customers_silver = customers
customers_rejected = pl.concat([
    rej_null.with_columns(pl.lit("null_customer_id").alias("reject_reason")),
    rej_dup.with_columns(pl.lit("duplicate_customer_id").alias("reject_reason")),
], how="diagonal")

write_parquet(customers_silver, config.SILVER_DIR / "customers.parquet")
write_parquet(customers_rejected, config.QUARANTINE_DIR / "customers.parquet")

In [6]:
quality_rows = []
def record_check(check_name, table, column, checked, failed, stage):
    quality_rows.append({
        "check_name": check_name, "table": table, "column": column,
        "records_checked": checked, "records_failed": failed,
        "pct_failed": round(100 * failed / checked, 2) if checked else 0.0,
        "stage": stage, "executed_at": datetime.now(),
    })

null_counts = customers_bronze.select(["customer_name", "email", "city", "state"]).null_count()
for col in ["customer_name", "email", "city", "state"]:
    record_check("null_check", "customers", col, customers_bronze.height, null_counts[col][0], "bronze")
record_check("null_check", "customers", "customer_id", customers_bronze.height, rej_null.height, "bronze")
record_check("duplicate_check", "customers", "customer_id", customers_bronze.height, rej_dup.height, "bronze")
record_check("row_count", "customers", "*", customers_bronze.height, customers_bronze.height - customers_silver.height, "silver")

quality = pl.DataFrame(quality_rows)
write_parquet(quality, config.QUALITY_DIR / "customers.parquet")
quality

check_name,table,column,records_checked,records_failed,pct_failed,stage,executed_at
str,str,str,i64,i64,f64,str,datetime[μs]
"""null_check""","""customers""","""customer_name""",500,9,1.8,"""bronze""",2026-07-09 23:41:56.831619
"""null_check""","""customers""","""email""",500,16,3.2,"""bronze""",2026-07-09 23:41:56.831625
"""null_check""","""customers""","""city""",500,6,1.2,"""bronze""",2026-07-09 23:41:56.831628
"""null_check""","""customers""","""state""",500,1,0.2,"""bronze""",2026-07-09 23:41:56.831629
"""null_check""","""customers""","""customer_id""",500,0,0.0,"""bronze""",2026-07-09 23:41:56.831650
"""duplicate_check""","""customers""","""customer_id""",500,0,0.0,"""bronze""",2026-07-09 23:41:56.831662
"""row_count""","""customers""","""*""",500,0,0.0,"""silver""",2026-07-09 23:41:56.831673


In [7]:
try:
    Customers.validate(customers_silver, lazy=True)
    print("Customers silver table passed")
except pa.errors.SchemaErrors as e:
    display(e.failure_cases)

Customers silver table passed


Products

In [8]:
products = read_csv_raw("products")
s_products = products.sample(fraction=0.1, seed=42)
s_products.head(5)

product_id,product_name,category,price,weight_kg
str,str,str,str,str
"""PROD-0023""","""Office Supplies Item 23""","""office_supplies""","""4131.69""","""11.46"""
"""PROD-0024""","""Books Item 24""","""books""","""4123.64""",null
"""PROD-0067""","""Home Appliances Item 67""","""home_appliances""","""103.3""","""19.7"""
"""PROD-0075""","""Home Appliances Item 75""","""home_appliances""","""3558.07""","""36.67"""
"""PROD-0082""","""Furniture Item 82""",null,"""2470.33""","""20.06"""


In [9]:
class Products(pa.DataFrameModel):
    product_id: str = pa.Field(nullable=False, unique=True)
    product_name: str = pa.Field(nullable=True)
    category: str = pa.Field(nullable=True, isin=list(config.PRODUCT_CATEGORIES))  # sello redundante
    price: float = pa.Field(nullable=True, ge=0)
    weight_kg: float = pa.Field(nullable=True, ge=0)
    source_file: str = pa.Field(alias="_source_file", nullable=False)
    ingested_at: datetime = pa.Field(alias="_ingested_at", nullable=False)

    class Config:
        coerce = False
        strict = True

In [10]:
products_bronze = s_products.with_columns(
    pl.lit("products.csv").alias("_source_file"),
    pl.lit(datetime.now()).alias("_ingested_at")
)
write_parquet(products_bronze, config.BRONZE_DIR / "products.parquet")

### Explorar categorías → derivar el set de valores aceptados

Antes de validar `category` contra una lista de valores permitidos, hay que **saber cuáles son**. Aquí exploramos las categorías presentes en el crudo y las enumeramos.

**Suposición:** derivar la lista directamente de los datos **de esta forma** solo es válido porque este dataset es **pequeño y fijo**, así que se pueden ver todas fácilmente. En un caso real la lista de categorías válidas debería venir de las **reglas de negocio**, de una **tabla de dimensión** o de una exploración mas optimizada. La lista final (derivada de esta exploración) vive en `config.PRODUCT_CATEGORIES` y se usa en el `accepted_values` check.

In [11]:
categorias = sorted(read_csv_raw("products")["category"].drop_nulls().unique().to_list())
print(f"{len(categorias)} categorías encontradas:")
categorias

15 categorías encontradas:


['automotive',
 'beauty',
 'books',
 'clothing',
 'electronics',
 'food',
 'furniture',
 'garden',
 'health',
 'home_appliances',
 'music',
 'office_supplies',
 'pet_shop',
 'sports',
 'toys']

In [12]:
products = strip_strings(products_bronze, ["product_id", "product_name", "category"])
products = lowercase(products, "category")
products, cast_price = cast_float(products, "price")
products, cast_weight = cast_float(products, "weight_kg")
products, neg_price = nullify_outside_range(products, "price", low=0)
products, rej_cat = nullify_not_in_set(products, "category", config.PRODUCT_CATEGORIES)

products, rej_null = drop_null_keys(products, "product_id")
products, rej_dup = deduplicate_by_key(products, "product_id")

products_silver = products
products_rejected = pl.concat([
    rej_null.with_columns(pl.lit("null_product_id").alias("reject_reason")),
    rej_dup.with_columns(pl.lit("duplicate_product_id").alias("reject_reason")),
    rej_cat.with_columns(pl.lit("invalid_category").alias("reject_reason")),
], how="diagonal")

write_parquet(products_silver, config.SILVER_DIR / "products.parquet")
write_parquet(products_rejected, config.QUARANTINE_DIR / "products.parquet")

In [13]:
quality_rows = []
def record_check(check_name, table, column, checked, failed, stage):
    quality_rows.append({
        "check_name": check_name, "table": table, "column": column,
        "records_checked": checked, "records_failed": failed,
        "pct_failed": round(100 * failed / checked, 2) if checked else 0.0,
        "stage": stage, "executed_at": datetime.now(),
    })

null_counts = products_bronze.select(["product_name", "category", "price", "weight_kg"]).null_count()
for col in ["product_name", "category", "price", "weight_kg"]:
    record_check("null_check", "products", col, products_bronze.height, null_counts[col][0], "bronze")
record_check("null_check", "products", "product_id", products_bronze.height, rej_null.height, "bronze")
record_check("duplicate_check", "products", "product_id", products_bronze.height, rej_dup.height, "bronze")

record_check("cast_check", "products", "price", products_bronze.height, cast_price, "silver")
record_check("cast_check", "products", "weight_kg", products_bronze.height, cast_weight, "silver")
record_check("range_check", "products", "price", products_bronze.height, neg_price, "silver")
record_check("accepted_values", "products", "category", products_bronze.height, rej_cat.height, "silver")

quality = pl.DataFrame(quality_rows)
write_parquet(quality, config.QUALITY_DIR / "products.parquet")
quality

check_name,table,column,records_checked,records_failed,pct_failed,stage,executed_at
str,str,str,i64,i64,f64,str,datetime[μs]
"""null_check""","""products""","""product_name""",100,0,0.0,"""bronze""",2026-07-09 23:41:56.871228
"""null_check""","""products""","""category""",100,4,4.0,"""bronze""",2026-07-09 23:41:56.871233
"""null_check""","""products""","""price""",100,0,0.0,"""bronze""",2026-07-09 23:41:56.871235
"""null_check""","""products""","""weight_kg""",100,2,2.0,"""bronze""",2026-07-09 23:41:56.871236
"""null_check""","""products""","""product_id""",100,0,0.0,"""bronze""",2026-07-09 23:41:56.871251
"""duplicate_check""","""products""","""product_id""",100,0,0.0,"""bronze""",2026-07-09 23:41:56.871261
"""cast_check""","""products""","""price""",100,0,0.0,"""silver""",2026-07-09 23:41:56.871270
"""cast_check""","""products""","""weight_kg""",100,0,0.0,"""silver""",2026-07-09 23:41:56.871278
"""range_check""","""products""","""price""",100,2,2.0,"""silver""",2026-07-09 23:41:56.871288


In [14]:
try:
    Products.validate(products_silver, lazy=True)
    print("Products silver table passed")
except pa.errors.SchemaErrors as e:
    display(e.failure_cases)

Products silver table passed
